# Wikipedia Retriever

In [1]:
from langchain_community.retrievers import WikipediaRetriever
retriver =WikipediaRetriever(top_k_results=2,lang='en')


In [2]:
query="The history of the how the sequence to sequence models are improved"
docs=retriver.invoke(query)

In [3]:
print(docs)

[Document(metadata={'title': 'The Simpsons opening sequence', 'summary': 'The Simpsons opening sequence is the title sequence of the American animated television series The Simpsons. It is accompanied by "The Simpsons Theme". The first episode to use this introduction was the series\' second episode "Bart the Genius".\nEach episode has the same basic sequence of events: the camera zooms through cumulus clouds, through the show\'s title towards the town of Springfield. The camera then follows the members of the Simpson family on their way home. Upon entering their house, the Simpsons settle down on their couch to watch television. One of the most distinctive aspects of the opening is that three of its elements change from episode to episode: Bart writes different phrases on the school chalkboard, Lisa plays different solos on her saxophone (or occasionally a different instrument), and different visual gags accompany the family as they enter their living room to sit on the couch.\nThe st

In [4]:
query="how actually the iran and israel war start in 5 lines"
docs2=retriver.invoke(query)

In [5]:
print(docs2)

[Document(metadata={'title': 'Iran', 'summary': "Iran, officially the Islamic Republic of Iran, and also known as Persia, is a country in West Asia. It borders Iraq to the west, Turkey, Azerbaijan, and Armenia to the northwest, the Caspian Sea to the north, Turkmenistan to the northeast, Afghanistan to the east, Pakistan to the southeast, and the Gulf of Oman and the Persian Gulf to the south. With a population of over 92 million, Iran ranks 17th globally in both geographic size and population and is the sixth-largest country in Asia. It is divided into five regions with 31 provinces. Tehran is the nation's capital, largest city, and financial center.\nHome to one of the world's oldest continuous major civilizations, Iran was first united as a nation by the Medes under Cyaxares in the 7th century BC and reached its territorial height in the 6th century BC, when Cyrus the Great founded the Achaemenid Empire. Alexander the Great conquered the empire in the 4th century BC. An Iranian rebe

In [6]:
print(type(docs2))

<class 'list'>


In [7]:
print(docs2[0])

page_content='Iran, officially the Islamic Republic of Iran, and also known as Persia, is a country in West Asia. It borders Iraq to the west, Turkey, Azerbaijan, and Armenia to the northwest, the Caspian Sea to the north, Turkmenistan to the northeast, Afghanistan to the east, Pakistan to the southeast, and the Gulf of Oman and the Persian Gulf to the south. With a population of over 92 million, Iran ranks 17th globally in both geographic size and population and is the sixth-largest country in Asia. It is divided into five regions with 31 provinces. Tehran is the nation's capital, largest city, and financial center.
Home to one of the world's oldest continuous major civilizations, Iran was first united as a nation by the Medes under Cyaxares in the 7th century BC and reached its territorial height in the 6th century BC, when Cyrus the Great founded the Achaemenid Empire. Alexander the Great conquered the empire in the 4th century BC. An Iranian rebellion in the 3rd century BC establis

In [8]:
# Print retrieved content
for i, doc in enumerate(docs2):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
Iran, officially the Islamic Republic of Iran, and also known as Persia, is a country in West Asia. It borders Iraq to the west, Turkey, Azerbaijan, and Armenia to the northwest, the Caspian Sea to the north, Turkmenistan to the northeast, Afghanistan to the east, Pakistan to the southeast, and the Gulf of Oman and the Persian Gulf to the south. With a population of over 92 million, Iran ranks 17th globally in both geographic size and population and is the sixth-largest country in Asia. It is divided into five regions with 31 provinces. Tehran is the nation's capital, largest city, and financial center.
Home to one of the world's oldest continuous major civilizations, Iran was first united as a nation by the Medes under Cyaxares in the 7th century BC and reached its territorial height in the 6th century BC, when Cyrus the Great founded the Achaemenid Empire. Alexander the Great conquered the empire in the 4th century BC. An Iranian rebellion in the 3rd centur

# 2 vector store and retriver

In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
load_dotenv()


# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vector_store = Chroma.from_documents(
        
        documents=documents,
        embedding=embeddings,
        collection_name="my_collection"
        
    )


In [10]:
retriver = vector_store.as_retriever(search_kwargs={"k":2})


In [11]:
query2 ="Why we used chroma bd ?"
results=retriver.invoke(query2)


In [12]:
for i , docs in enumerate(results):
    print(f"\n ---- Results {i+1}")
    print(docs.page_content)


 ---- Results 1
Chroma is a vector database optimized for LLM-based search.

 ---- Results 2
LangChain helps developers build LLM applications easily.


In [13]:
results3= vector_store.similarity_search(query2, k=2)

In [14]:
for i , docs in enumerate(results3):
    print(f"\n --Result{i+1}-----")
    print(docs.page_content)


 --Result1-----
Chroma is a vector database optimized for LLM-based search.

 --Result2-----
LangChain helps developers build LLM applications easily.


# 3 MMR Maximal Marginal Relevance

In [15]:

# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [16]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_model =GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [17]:

# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 1}  # k = top results, lambda_mult = relevance-diversity balance
)

In [18]:
query="what is langchain?"
results = retriver.invoke(query)

In [19]:
print(type(results))
print(results)

<class 'list'>
[Document(id='737f7dd2-eab6-414e-b07e-b2e76693714c', metadata={}, page_content='LangChain helps developers build LLM applications easily.'), Document(id='1b0772b3-50e4-425c-b085-6d81943ebd45', metadata={}, page_content='OpenAI provides powerful embedding models.')]


In [20]:
print(results[0])

page_content='LangChain helps developers build LLM applications easily.'


In [21]:

for doc in results:
    print(doc.page_content)

LangChain helps developers build LLM applications easily.
OpenAI provides powerful embedding models.


In [22]:
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [23]:

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain helps developers build LLM applications easily.

--- Result 2 ---
OpenAI provides powerful embedding models.


# 4 Multy Retriver

In [24]:

from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_classic.retrievers import MultiQueryRetriever

In [25]:
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [26]:
embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


In [27]:
vectorstore= FAISS.from_documents(
    documents=all_docs,
    embedding= embedding_model
)

In [28]:
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [29]:
multiquery_retriver=MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k":5}),
    llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash')
)

In [30]:
query = "How to improve energy levels and maintain balance?"

In [31]:
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriver.invoke(query)

In [32]:

for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 5 ---
Regular walking boosts heart health and can reduce symptoms of depression.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and impro

# 5 Contexual Compression Retreiver 

In [34]:

from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_core.documents import Document

from langchain_classic.retrievers import ContextualCompressionRetriever


In [35]:
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [36]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [37]:
embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


In [38]:
vectorstore = FAISS.from_documents(docs, embedding_model)

In [39]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


In [48]:
llm = ChatGoogleGenerativeAI(model= 'gemini-2.5-flash-lite')


In [49]:
compressor = LLMChainExtractor.from_llm(llm)

In [50]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [51]:

# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [52]:

for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
Photosynthesis does not occur in animal cells.

--- Result 3 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
